# 03 — Experimental Design: CV Strategy × Model × Representation

**Research question:** Do algorithm and feature representation rankings change when validation is stratified-random versus scaffold-separated?

**Safety:** Reconstruct the *same* existing locked research test with seed 42; run experiments **only on original TRAIN** (6,841 expected) so original development validation (1,074) stays reserved for later threshold and candidate decisions. Never score the locked test here.

**How to run:** Run cells top to bottom. Start with `RUN_NESTED=False` to profile the selected pre-tuned configurations; then set `RUN_NESTED=True` and rerun for a fully tuned within-outer-fold comparison. The first is a **sensitivity diagnostic**, not an unbiased estimate of optimized model selection. All artifacts save to `research_outputs/experiment_03/`.

## 0. Imports, resource and run controls
Keep parallelism bounded on shared UCSF Jupyter resources. No GPU required for this notebook.

In [ ]:
from pathlib import Path
import ast, json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, StratifiedKFold, StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score
import sklearn, rdkit

SEED=42
RUN_NESTED=False     # Change to True only after reviewing quick-run timing
OUT=Path('research_outputs/experiment_03'); OUT.mkdir(parents=True,exist_ok=True)
DATA=Path('hw1data.tab')
PREVIOUS=Path('research_outputs/coarse_cv.csv')
assert DATA.exists(), f'Missing {DATA}. Start Jupyter inside CDD203-HW1.'
print('Python', platform.python_version(), 'sklearn', sklearn.__version__, 'RDKit', rdkit.__version__)
print('Nested tuning enabled?', RUN_NESTED)
RDLogger.DisableLog('rdApp.warning')

## 1. Recover original splits exactly (never remake locked test)
The previous notebook used two successive `GroupShuffleSplit` calls. This code repeats their exact method and verifies non-overlap of scaffolds. **GroupShuffleSplit fractions refer to groups, not rows.**

In [ ]:
df=pd.read_csv(DATA,sep='\t')
assert {'Drug','Y'}.issubset(df.columns),df.columns.tolist()
assert df['Y'].isin([0,1]).all() and df['Drug'].notna().all()
molecules=[Chem.MolFromSmiles(s) for s in df['Drug']]
assert all(m is not None for m in molecules)
groups=np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m),canonical=True) for m in molecules])
y=df['Y'].to_numpy(dtype=int)
idx=np.arange(len(df))
trainval_idx,test_idx=next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED).split(idx,y,groups))
rel_train,rel_val=next(GroupShuffleSplit(n_splits=1,test_size=.125,random_state=SEED).split(trainval_idx,y[trainval_idx],groups[trainval_idx]))
train_idx=trainval_idx[rel_train]; val_idx=trainval_idx[rel_val]
for label,ii in [('TRAIN',train_idx),('DEV VALIDATION',val_idx),('LOCKED TEST',test_idx)]:
    print(label,len(ii),'positive fraction',round(float(y[ii].mean()),3),'unique scaffolds',len(set(groups[ii])))
assert all(set(groups[a]).isdisjoint(set(groups[b])) for a,b in [(train_idx,val_idx),(train_idx,test_idx),(val_idx,test_idx)])
assert len(df)==10000 and len(train_idx)==6841 and len(val_idx)==1074 and len(test_idx)==2085, 'Split disagrees with recorded previous notebook! STOP and investigate.'
# Use original training only. Neither validation nor locked test feeds the CV experiments.
work_idx=train_idx.copy(); yw=y[work_idx]; gw=groups[work_idx]
assert len(np.unique(yw))==2
print('SAFE: only original TRAIN will enter the CV experiments.')

## 2. Rebuild original representations exactly
Morgan: radius 2, 2048 binary bits; 10 deterministic descriptors; Hybrid concatenates fingerprints and descriptors. Feature computation depends only on molecular structure, never labels.

In [ ]:
def descriptors10(mols):
    rows=[]
    for m in mols:
        rows.append([Descriptors.MolWt(m),Descriptors.MolLogP(m),rdMolDescriptors.CalcTPSA(m),
            rdMolDescriptors.CalcNumHBD(m),rdMolDescriptors.CalcNumHBA(m),
            rdMolDescriptors.CalcNumRotatableBonds(m),rdMolDescriptors.CalcNumAromaticRings(m),
            rdMolDescriptors.CalcFractionCSP3(m),Chem.GetFormalCharge(m),
            sum(a.GetAtomicNum()==7 for a in m.GetAtoms())])
    return np.asarray(rows,dtype=np.float32)

def morgan_matrix(mols,radius=2,bits=2048):
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=radius,fpSize=bits)
    result=np.zeros((len(mols),bits),dtype=np.float32)
    for i,m in enumerate(mols):
        DataStructs.ConvertToNumpyArray(gen.GetFingerprint(m),result[i])
    return result

t=time.perf_counter()
training_mols=[molecules[i] for i in work_idx]
X_desc=descriptors10(training_mols)
X_morgan=morgan_matrix(training_mols)
REPS={'Morgan':X_morgan,'Descriptors10':X_desc,'Hybrid':np.hstack([X_morgan,X_desc]).astype(np.float32)}
print('Featurization seconds:',round(time.perf_counter()-t,2))
for name,x in REPS.items():
    assert len(x)==len(yw) and np.isfinite(x).all()
    print(name,x.shape, 'MiB',round(x.nbytes/2**20,1))

## 3. CV designs + fold diagnostics
We compare *StratifiedKFold* (classes balanced, shared scaffolds permitted) and *StratifiedGroupKFold* (scaffolds separated, approximate class balance). For provenance, also note that prior coarse results used **GroupKFold(3)**; thus exact historical scores are **not** directly comparable to our new fold-level results.

In [ ]:
N_OUTER=3  # Pilot: 3; change to 5 after checking runtime and fold quality
OUTER_CV={
 'Stratified':StratifiedKFold(n_splits=N_OUTER,shuffle=True,random_state=SEED),
 'Scaffold_stratified_group':StratifiedGroupKFold(n_splits=N_OUTER,shuffle=True,random_state=SEED)
}
def get_folds(method, n_splits=N_OUTER, seed=SEED):
    if method=='Stratified':
        splitter=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)
        return list(splitter.split(np.zeros(len(yw)),yw))
    splitter=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    return list(splitter.split(np.zeros(len(yw)),yw,gw))

fold_rows=[]
for method in OUTER_CV:
    for j,(tr,va) in enumerate(get_folds(method),1):
        overlap=len(set(gw[tr]) & set(gw[va]))
        fold_rows.append({'CV':method,'Fold':j,'train_n':len(tr),'valid_n':len(va),
                          'train_positive':yw[tr].mean(),'valid_positive':yw[va].mean(),
                          'shared_scaffolds':overlap})
        assert len(np.unique(yw[tr]))==2 and len(np.unique(yw[va]))==2
        if method!='Stratified': assert overlap==0
fold_diagnostics=pd.DataFrame(fold_rows)
display(fold_diagnostics.round(3))
fold_diagnostics.to_csv(OUT/'fold_diagnostics.csv',index=False)

## 4. Candidate models and previous tuned settings
The old coarse search tuned on original TRAIN under GroupKFold(3). We can reuse best settings **only for a cheap diagnostic**. Reusing them for new CV scores does not create an independent unbiased performance estimate. Nested mode below retunes settings *inside each outer training fold*.

In [ ]:
assert PREVIOUS.exists(),f'Missing {PREVIOUS}; supply previous coarse_cv.csv under research_outputs.'
previous=pd.read_csv(PREVIOUS)
CANDIDATES=[('Hybrid','Random Forest'),('Hybrid','Extra Trees'),('Hybrid','SVM-RBF')]
for rep,model in CANDIDATES:
    assert ((previous.Representation==rep)&(previous.Model==model)).any()

def model_factory(name,rep,p):
    if name=='SVM-RBF':
        if rep=='Descriptors10': pre=StandardScaler()
        elif rep=='Hybrid': pre=ColumnTransformer([('desc',StandardScaler(),list(range(p-10,p)))],remainder='passthrough')
        else:pre='passthrough'
        return Pipeline([('pre',pre),('model',SVC(kernel='rbf',probability=True,random_state=SEED))])
    if name=='Random Forest':return RandomForestClassifier(random_state=SEED,n_jobs=2)
    if name=='Extra Trees':return ExtraTreesClassifier(random_state=SEED,n_jobs=2)
    raise ValueError(name)

previous_params={}
for rep,name in CANDIDATES:
    row=previous[(previous.Representation==rep)&(previous.Model==name)].iloc[0]
    previous_params[(rep,name)]=ast.literal_eval(row.Settings)
    print(rep,name,previous_params[(rep,name)])
# Bounded grids for nested mode, comparable in intent rather than identical numerical settings.
GRIDS={
 'Random Forest': {'n_estimators':[160], 'min_samples_leaf':[1,3,5], 'max_features':['sqrt',.2]},
 'Extra Trees': {'n_estimators':[160], 'min_samples_leaf':[1,3,5], 'max_features':['sqrt',.2]},
 'SVM-RBF': {'model__C':[.1,1,10], 'model__gamma':['scale',.01]}
}

## 5. Evaluate CV × model with optional **nested hyperparameter tuning**
**Quick mode (`RUN_NESTED=False`):** Fit previous tuned settings on each outer training fold; use for split-sensitivity screening and timing only. **Nested mode (`True`):** Inside each outer training fold, retune using an *inner 3-fold CV of the same validation family*, then score the untouched outer fold. Inner preprocessing (SVM descriptor scaling) is inside its pipeline. We do **not** select thresholds from any outer fold.

This study initially holds Hybrid fixed for interpretability. Follow up with the Morgan/Descriptor/Hybrid interaction only after profiling runtime.

In [ ]:
INNER_FOLDS=3
RUN_REPRESENTATION_EXTENSION=False # Enables Morgan and Descriptors10 for each model later
candidates=([(rep,name) for name in ['Random Forest','Extra Trees','SVM-RBF']
             for rep in (['Hybrid','Morgan','Descriptors10'] if RUN_REPRESENTATION_EXTENSION else ['Hybrid'])])

results=[]
start_all=time.perf_counter()
for method in OUTER_CV:
    for rep,name in candidates:
        X=REPS[rep]; folds=get_folds(method)
        for fold_id,(tr,va) in enumerate(folds,1):
            est=model_factory(name,rep,X.shape[1]);tic=time.perf_counter()
            if RUN_NESTED:
                if method=='Stratified': inner=StratifiedKFold(n_splits=INNER_FOLDS,shuffle=True,random_state=SEED)
                else:inner=StratifiedGroupKFold(n_splits=INNER_FOLDS,shuffle=True,random_state=SEED)
                search=GridSearchCV(est,GRIDS[name],cv=inner,scoring='roc_auc',refit=True,
                                    n_jobs=1,error_score='raise')
                fit_kw={'groups':gw[tr]} if method!='Stratified' else {}
                search.fit(X[tr],yw[tr],**fit_kw)
                fitted=search.best_estimator_;params=search.best_params_
            else:
                if (rep,name) in previous_params:
                    params=previous_params[(rep,name)]
                else:
                    orig=previous[(previous.Representation==rep)&(previous.Model==name)]
                    if orig.empty: raise ValueError(f'No previously tuned setting for {rep}/{name}; use RUN_NESTED=True')
                    params=ast.literal_eval(orig.iloc[0].Settings)
                fitted=est.set_params(**params).fit(X[tr],yw[tr])
            p=fitted.predict_proba(X[va])[:,1]
            elapsed=time.perf_counter()-tic
            results.append({'CV':method,'Representation':rep,'Model':name,'Fold':fold_id,
               'AUROC':roc_auc_score(yw[va],p),'AUPRC':average_precision_score(yw[va],p),
               'Seconds':elapsed,'n_train':len(tr),'n_valid':len(va),
               'positive_rate':yw[va].mean(),'parameters':json.dumps(params,default=str),
               'NestedTuned':RUN_NESTED})
            print(f'{method:28s} {rep:14s} {name:14s} fold={fold_id} AUROC={results[-1]["AUROC"]:.4f} sec={elapsed:.1f}')
print('Total wall-clock min:',round((time.perf_counter()-start_all)/60,2))
fold_results=pd.DataFrame(results)
suffix='nested' if RUN_NESTED else 'fixed_prior_settings'
fold_results.to_csv(OUT/f'fold_scores_{suffix}.csv',index=False)

## 6. Summary, visualization, and cautious interpretation
Compare average AUROC/AUPRC and variability across outer folds. Changing split changes the evaluation problem; differences are **not** causal effects of a splitting algorithm. Fold-level SD with only 3 folds is descriptive. No p-values or winner claims.

In [ ]:
summary=(fold_results.groupby(['CV','Representation','Model'],as_index=False)
         .agg(mean_AUROC=('AUROC','mean'),sd_AUROC=('AUROC','std'),
              mean_AUPRC=('AUPRC','mean'),total_seconds=('Seconds','sum')))
display(summary.sort_values(['Model','CV']).round(4))
summary.to_csv(OUT/f'summary_{suffix}.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4))
for i,method in enumerate(summary.CV.unique()):
    sub=summary[summary.CV==method]
    x=np.arange(len(sub))
    ax.errorbar(x+i*.05,sub.mean_AUROC,yerr=sub.sd_AUROC,fmt='o',label=method,capsize=3)
ax.set_xticks(np.arange(len(sub)))
ax.set_xticklabels((sub['Model']+' / '+sub['Representation']).tolist(),rotation=30,ha='right')
ax.set_ylabel('Outer-fold AUROC'); ax.legend();plt.tight_layout();plt.show()
print('Do not inspect locked test here. Next step: review fold diagnostics and runtime; then nested tuned run if feasible.')

## 7. Predefined decision rules and next steps

1. **Validity first:** check each fold has both labels and group overlap is zero for scaffold-aware folds. Compare class proportions and training sizes.
2. **Compute:** check wall-clock time and memory. Only scale outer folds from 3 to 5 if feasible.
3. **Fairness:** when judging final candidate rankings, prefer `RUN_NESTED=True` with tuning nested in each outer fold. Fixed-prior-settings scores are exploratory only.
4. **Model × representation interaction:** turn on `RUN_REPRESENTATION_EXTENSION=True` for a bounded follow-up if initial results justify it; in nested mode all representations are tuned separately.
5. **Selection:** decide the operational priority (e.g. screening sensitivity subject to specificity constraint) *before* selecting threshold. Optimize thresholds using development validation or inner/OOF predictions, **not** outer test folds being used to report final performance.
6. **Holdouts:** original dev validation (1,074) and locked research test (2,085) have not been used by any training or scoring cell here. Historical notebook's development validation may already have influenced decisions, so it is not necessarily pristine. Research locked test stays unopened until final freeze.
7. **Submission:** only after choosing a finalized approach, integrate `featurize`, `build_model`, `THRESHOLD` into professor's `Part1_TEMPLATE.py`; run `TestModel.py` and check ONNX constraints separately. This notebook is for research, **not** the final submission.